# Phase 2: Can LoRA recover accuracy lost to weight-only quantization?

**Model:** `meta-llama/Llama-3.2-1B` &nbsp;|&nbsp; **Grid:** 3 bases (FP16, INT8-RTN, INT4-RTN) x 3 seeds (0, 1, 2) = 9 LoRA runs, plus 3 no-LoRA baselines.

**What this notebook does, step by step**

1. Installs pinned libraries and sets one `CONFIG` that every run shares.
2. Mounts Google Drive (all outputs go to `MyDrive/quant-study-lora/`) and logs in to Hugging Face (the model is gated).
3. Rebuilds the Phase 1 quantized models *in PyTorch* by **fake-quantizing** the weights with the same math NNCF 3.4.0 uses (RTN: round-to-nearest, no calibration data).
4. Runs a **sanity check**: WikiText perplexity of the fake-quant INT4 model vs FP16.
5. Trains LoRA adapters (r=16) on a fixed 5,000-example Dolly subset, once per (base, seed). Each finished run writes a `DONE` file, so after a Colab disconnect you can just re-run the notebook and it will skip finished runs.
6. Evaluates every adapter **and the three no-LoRA bases** with `lm-evaluation-harness` (zero-shot ARC-C, ARC-E, HellaSwag, WinoGrande, BoolQ, PIQA).
7. Aggregates everything into one table (mean ± std over seeds).

**Runtime:** Runtime → Change runtime type → **T4 GPU**. See the README for time estimates and a suggested split across sessions.

## Why fake quantization, and why re-evaluate the bases here?

Phase 1 produced OpenVINO IR models. PyTorch cannot backpropagate through an OpenVINO model, so we cannot train LoRA on those files directly.
Instead, we load the original FP16 Hugging Face weights and replace each weight matrix `W` with `dequantize(quantize(W))`, using NNCF's RTN formulas.
The resulting weights take exactly the same small set of values (e.g. 16 levels per group of 128 for INT4) as the OpenVINO model, but they are stored as ordinary FP16 tensors that PyTorch can run.
The base weights are **frozen**; only the LoRA adapters (small extra matrices added beside `q_proj/k_proj/v_proj/o_proj`) are trained. This is the same idea as QLoRA. The adapters are never merged into the quantized weights.

**This is an approximation of the OpenVINO models, not the same thing.** The weight values should match NNCF's very closely, but the kernels differ (CUDA FP16 here vs OpenVINO CPU), and so do some numerical details such as accumulation precision and the inference precision hint. So accuracy numbers from this notebook can differ a little from the Phase 1 laptop numbers.

**Fairness:** for that reason, *"LoRA on"* is compared against *"LoRA off"* measured **in this same Colab simulation** (same fake-quant weights, same lm-eval version, same GPU kernels, same batch settings). The only thing that changes is whether the adapter is attached. The Phase 1 OpenVINO numbers are reported alongside for context, but the recovery claim should rest on the within-Colab comparison.

## 1. Install libraries

Versions are pinned loosely (tested on the box with transformers 4.57, peft 0.21, lm-eval 0.4.13, datasets 5.0, accelerate 1.15). Colab's preinstalled PyTorch is used as-is. If pip downgrades transformers, use **Runtime → Restart session** once and then continue from the next cell.

In [ ]:
%pip uninstall -y -q torchao  # Colab ships an old torchao that newer PEFT rejects; not used here (restart session after first install)
%pip install -q "transformers>=4.56,<5" "peft>=0.13,<1" "accelerate>=1.0" "datasets>=3.0,<6" "lm-eval>=0.4.8,<0.5" "pandas"

## 2. CONFIG: every run uses these settings

Nothing below this cell should change between runs. Only `base` and `seed` vary across the grid, and both are recorded in each run's `run_config.json`.
To split the work across Colab sessions, edit `SESSION_BASES` / `SESSION_SEEDS` (the default runs whatever is not finished yet).

In [ ]:
CONFIG = {
    # ---- model / study grid ----
    "model_id": "meta-llama/Llama-3.2-1B",
    "bases": ["fp16", "int8-rtn", "int4-rtn"],
    "seeds": [0, 1, 2],

    # ---- fake-quant scheme: matches the Phase 1 IRs (models\ov-rtn-int8, models\ov-rtn-int4) exactly ----
    # Verified from the IRs' NNCF rt_info and by bit-exact comparison of stored codes/scales (NNCF 3.4.0,
    # optimum 2.3.0 / optimum-intel 2.2.0, data-free RTN: awq/scale_estimation/gptq all False).
    # int8-rtn: mode=INT8_SYM (i8, no zero point), per-output-channel, ALL weights incl. the tied embedding/lm_head.
    #           (NNCF's own default would be INT8_ASYM; Phase 1 used symmetric, presumably `--weight-format int8 --sym`.)
    # int4-rtn: mode=INT4_SYM, group_size=128, ratio=1.0, all_layers=False -> all 112 decoder Linear layers INT4_SYM,
    #           tied embedding/lm_head kept in backup mode INT8_ASYM per-channel (u8 + u8 zero point).
    "int8_mode": "int8_sym",
    "int4_group_size": 128,
    "ov_fp16_scale": True,             # OpenVINO stores the scale (and zero point) as f16; mimic that rounding

    # ---- LoRA (fixed) ----
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],

    # ---- data (fixed; identical subset for every run) ----
    "dataset_id": "databricks/databricks-dolly-15k",
    "data_seed": 1234,                 # picks the subset; independent of the training seed
    "train_size": 5000,
    "holdout_size": 200,               # small held-out Dolly set for a final eval loss
    "max_seq_len": 512,                # examples longer than this are dropped *before* sampling (no truncation)
    "loss_on_response_only": True,

    # ---- training (fixed) ----
    "epochs": 1,
    "learning_rate": 2e-4,
    "lr_scheduler": "cosine",
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "per_device_batch_size": 4,
    "grad_accum_steps": 4,             # effective batch = 4 x 4 = 16
    "max_grad_norm": 1.0,
    "optimizer": "adamw_torch",
    "gradient_checkpointing": True,
    "logging_steps": 10,
    # T4 has no bf16: frozen base in fp16, LoRA params in fp32, fp16 autocast + loss scaling.
    "fp16_autocast": True,

    # ---- evaluation ----
    "eval_tasks": ["arc_challenge", "arc_easy", "hellaswag", "winogrande", "boolq", "piqa"],
    "eval_num_fewshot": 0,
    "eval_batch_size": "auto",       # lm-eval picks the largest batch that fits; use 8 if you hit OOM. Future runs should fix this (e.g. 11 or 13): auto batch size can change with free GPU memory and flip near-tie questions (see the paper, Section 4.3 and Appendix Table 19).
    "eval_limit": None,                # None = full task sets (needed for the paper); e.g. 50 for a quick test
    "eval_add_bos": True,              # Llama-3 expects BOS; keep consistent with Phase 1 if it differed
    "sanity_wikitext_docs": 20,        # WikiText-2 test docs for the sanity ppl (None = all 62 docs)
    "sanity_max_length": 2048,         # rolling-window length, same as the laptop measurement

    # ---- storage ----
    "drive_root": "/content/drive/MyDrive/quant-study-lora",
}

# Metric per task, same as Phase 1.
TASK_METRIC = {
    "arc_challenge": "acc_norm", "arc_easy": "acc_norm", "hellaswag": "acc_norm",
    "piqa": "acc_norm", "winogrande": "acc", "boolq": "acc",
}

# Which runs to do in *this* session (None = all of them; finished runs are skipped anyway).
SESSION_BASES = None   # e.g. ["fp16"]
SESSION_SEEDS = None   # e.g. [0, 1]

# Phase 1 laptop reference (OpenVINO, with-BOS word perplexity, 2048-token windows). For context only.
PHASE1_WIKITEXT_PPL = {"fp16": 15.46, "int4-rtn": 19.98}

## 3. Mount Google Drive and log in to Hugging Face

`meta-llama/Llama-3.2-1B` is gated. Make sure your Hugging Face account has accepted the licence on the model page.
The preferred login is a Colab secret: click the key icon in the left sidebar, add a secret named `HF_TOKEN` (a read token from huggingface.co/settings/tokens), and enable notebook access.
If no secret is found, an interactive login box appears instead.

In [ ]:
import os
# Set USE_DRIVE = False if the Drive popup keeps failing. Results then go to the Colab disk
# (/content/quant-study-lora), and you MUST run the "download results" cell at the end of each
# session (and re-upload the zip next session), because the Colab disk is wiped on disconnect.
USE_DRIVE = True   # your Phase 2 results are on Drive, so keep this True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    CONFIG["drive_root"] = "/content/quant-study-lora"
    print("Saving to the Colab disk, NOT Drive. Download the zip before closing!")
os.makedirs(CONFIG["drive_root"], exist_ok=True)

from huggingface_hub import login, notebook_login, whoami
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    notebook_login()   # paste a token when the widget appears, then re-run this cell
print("Logged in to Hugging Face as:", whoami()["name"])

## 4. Imports and helpers

In [ ]:
# Guard: make PEFT ignore Colab's old torchao even if the uninstall didn't take effect
import sys as _sys
import peft.import_utils as _piu
_piu.is_torchao_available = lambda: False
for _n, _m in list(_sys.modules.items()):
    if _n.startswith("peft") and hasattr(_m, "is_torchao_available"):
        _m.is_torchao_available = lambda: False

import gc, json, math, time, hashlib, platform
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import transformers, peft, datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, TrainerCallback
from peft import LoraConfig, PeftModel, get_peft_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DRIVE = Path(CONFIG["drive_root"])

def run_name(base, seed):
    return f"{base}-seed{seed}"

def run_dir(base, seed):
    return DRIVE / run_name(base, seed)

def is_done(path):
    return (Path(path) / "DONE").exists()

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str))
    tmp.replace(path)   # atomic-ish: never leaves a half-written JSON on Drive

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def env_info():
    return {
        "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
        "datasets": datasets.__version__, "python": platform.python_version(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }

print(env_info())

## 5. Fake quantization that mirrors NNCF 3.4.0 RTN

These functions re-implement, in PyTorch, the data-free path of `nncf.compress_weights` (file `src/nncf/quantization/algorithms/weight_compression/weight_lowering.py` in NNCF v3.4.0):

* **INT8_SYM** (what the Phase 1 `ov-rtn-int8` model uses; NNCF's own default would be INT8_ASYM): one signed scale **per output channel** (each row of the `[out, in]` weight), no zero point.
  `scale = |min|/128 if |min| >= max else -max/128`, `q = clip(round(w / scale), -128, 127)`, `w' = q * scale`.
* **INT8_ASYM** (used for the embedding/lm_head of the INT4 model): one scale and zero point per output channel.
  `min = min(row, 0)`, `max = max(row, 0)`, `scale = (max - min) / 255`, `zero_point = clip(-round(min / scale), 0, 255)`, `q = clip(round(w / scale + zero_point), 0, 255)`, `w' = (q - zero_point) * scale`.
* **INT4_SYM, group size 128**: each row is split into groups of 128 input values. NNCF uses a *signed* scale so that the largest-magnitude value lands exactly on level -8:
  `scale = |min|/8 if |min| >= max else -max/8`, `q = clip(round(w / scale), -8, 7)`, `w' = q * scale`.
* **Which layers:** in INT4 mode with the defaults (`ratio=1`, `all_layers=False`), every decoder `Linear` becomes INT4_SYM, while the **embedding table and the last MatMul (lm_head)** stay in the backup precision, **INT8_ASYM per channel**. In Llama-3.2-1B the lm_head is tied to the embedding, so that one shared matrix is quantized once. In INT8 mode, all weights (including the embedding) are INT8_SYM.
* Rounding is round-half-to-even (same as NumPy and OpenVINO `Round`). Statistics and quantization are computed in FP32. OpenVINO then stores the scale as **f16**, so by default we round the scale to FP16 before dequantizing.

Norm weights (RMSNorm) are not compressed by NNCF and are left alone here too.

**Checked against the real Phase 1 files:** I read the NNCF settings stored inside `ov-rtn-int8` and `ov-rtn-int4` on the laptop. I also compared the integer codes, zero points, and FP16 scales that are stored there with this code applied to the same source weights (`models\fp16-src`, bf16). They are **100% identical** on every layer checked (q/k proj of layer 0, down_proj of layer 7, gate_proj of layer 15, embedding). Using the wrong mode (asym instead of sym) gives below 1% agreement. The only remaining differences come from the runtime (CUDA FP16 vs OpenVINO CPU kernels), not the weights.

In [ ]:
_EPS32 = torch.finfo(torch.float32).eps

def _grouped(w, group_size):
    """[out, in] -> [out, in // g, g] so stats are taken over the last axis (per group). -1 = per channel."""
    if group_size in (-1, None):
        return w.unsqueeze(1)                      # [out, 1, in]: one group = the whole row
    out_f, in_f = w.shape
    if in_f % group_size != 0:
        raise ValueError(f"in_features={in_f} not divisible by group_size={group_size}")
    return w.reshape(out_f, in_f // group_size, group_size)

def fake_quant_weight(w, mode, group_size=-1, fp16_scale=True):
    """Quantize-dequantize a 2-D weight [out, in] exactly like NNCF RTN. Returns FP32 tensor, same shape."""
    assert w.dim() == 2, w.shape
    orig_shape = w.shape
    wg = _grouped(w.detach().to(torch.float32), group_size)
    if mode in ("int8_asym", "int4_asym"):
        bits = 8 if mode == "int8_asym" else 4
        lo, hi = 0, 2 ** bits - 1
        mn = wg.amin(dim=-1, keepdim=True).clamp(max=0.0)
        mx = wg.amax(dim=-1, keepdim=True).clamp(min=0.0)
        scale = (mx - mn) / (hi - lo)
        scale = torch.where(scale.abs() < _EPS32, torch.full_like(scale, _EPS32), scale)
        zp = (lo - torch.round(mn / scale)).clamp(lo, hi)
        q = torch.round(wg / scale + zp).clamp(lo, hi)
    elif mode in ("int8_sym", "int4_sym"):
        bits = 8 if mode == "int8_sym" else 4
        lo, hi = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
        abs_min = wg.amin(dim=-1, keepdim=True).abs()
        mx = wg.amax(dim=-1, keepdim=True)
        scale = torch.where(abs_min >= mx, abs_min, -mx) / (2 ** (bits - 1))
        scale = torch.where(scale.abs() < _EPS32, torch.full_like(scale, _EPS32), scale)
        zp = None
        q = torch.round(wg / scale).clamp(lo, hi)
    else:
        raise ValueError(f"unknown mode {mode}")
    if fp16_scale:   # OpenVINO keeps scale / zero point constants in f16
        scale = scale.half().float()
    w_dq = (q - zp) * scale if zp is not None else q * scale
    return w_dq.reshape(orig_shape)

def quant_plan(model, base, cfg=CONFIG):
    """List of (name, weight_param, mode, group_size) following NNCF's layer assignment."""
    if base == "fp16":
        return []
    if base == "int8-rtn":
        primary, gs, backup = cfg["int8_mode"], -1, cfg["int8_mode"]
    elif base == "int4-rtn":
        primary, gs, backup = "int4_sym", cfg["int4_group_size"], "int8_asym"
    else:
        raise ValueError(base)
    plan, seen = [], set()
    emb = model.get_input_embeddings().weight
    plan.append(("model.embed_tokens", emb, backup, -1)); seen.add(id(emb))
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear) and ".layers." in name and id(mod.weight) not in seen:
            plan.append((name, mod.weight, primary, gs)); seen.add(id(mod.weight))
    head = model.get_output_embeddings().weight
    if id(head) not in seen:   # untied lm_head = "last MatMul" -> backup precision
        plan.append(("lm_head", head, backup, -1)); seen.add(id(head))
    return plan

@torch.no_grad()
def apply_fake_quant(model, base, cfg=CONFIG, device=DEVICE):
    """In-place: replace each planned weight with its NNCF-style quantize-dequantize value."""
    stats = []
    for name, p, mode, gs in quant_plan(model, base, cfg):
        w = p.data.to(device=device, dtype=torch.float32)
        w_dq = fake_quant_weight(w, mode, gs, cfg["ov_fp16_scale"])
        rel_err = ((w_dq - w).norm() / w.norm()).item()
        p.data.copy_(w_dq.to(p.device, p.dtype))
        stats.append({"name": name, "mode": mode, "group_size": gs, "shape": list(p.shape), "rel_err": rel_err})
    return stats

def load_base(base, cfg=CONFIG, device=DEVICE):
    """FP16 model on `device` with fake-quant weights for the given base. Base weights are frozen."""
    model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], dtype=torch.float32, low_cpu_mem_usage=True)
    model.to(device)                       # quantize from exact FP32 copies of the bf16 checkpoint
    stats = apply_fake_quant(model, base, cfg, device)
    if device == "cuda":
        model.half()                       # T4: FP16 storage for the frozen base
    model.requires_grad_(False)
    model.config.use_cache = False
    model.eval()
    return model, stats

def load_tokenizer(cfg=CONFIG):
    tok = AutoTokenizer.from_pretrained(cfg["model_id"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token      # pad positions are masked (-100 labels, attention_mask 0)
    return tok

### 5a. Quick unit check of the quantizer

On a random matrix: INT4 groups must use at most 16 distinct levels, INT8 rows at most 256, and errors must be small. This runs on CPU in seconds.

In [ ]:
torch.manual_seed(0)
_w = torch.randn(64, 256) * 0.02
for _mode, _gs, _levels in [("int4_sym", 128, 16), ("int8_asym", -1, 256), ("int8_sym", -1, 256)]:
    _dq = fake_quant_weight(_w, _mode, _gs, fp16_scale=False)
    _n = max(len(torch.unique(g)) for g in _grouped(_dq, _gs).reshape(-1, _w.shape[1] if _gs == -1 else _gs))
    print(f"{_mode:9s} gs={_gs:4d}  max distinct levels per group = {_n:3d} (limit {_levels})  "
          f"rel err = {((_dq - _w).norm() / _w.norm()).item():.4f}")
    assert _n <= _levels

## 6. Sanity check: WikiText perplexity, fake-quant INT4 vs FP16

This uses lm-evaluation-harness's `wikitext` task (WikiText-2, word-level perplexity, rolling 2048-token windows with a BOS prefix token).
It checks that the simulation degrades the model by a plausible amount. The laptop measured FP16 **15.46** and RTN INT4 **19.98** with OpenVINO; we only *report* our numbers next to these and do not tune anything to match them.
With `sanity_wikitext_docs = 20` this takes about 5 minutes per model. Set it to `None` for the full test split, which is the setup closest to the laptop's.

In [ ]:
import lm_eval
from lm_eval.models.huggingface import HFLM

def make_lm(model, tok, cfg=CONFIG, max_length=None, batch_size=None):
    return HFLM(pretrained=model, tokenizer=tok, batch_size=batch_size or cfg["eval_batch_size"],
                max_length=max_length, add_bos_token=cfg["eval_add_bos"])

def wikitext_ppl(model, tok, cfg=CONFIG):
    lm = make_lm(model, tok, cfg, max_length=cfg["sanity_max_length"], batch_size=1)
    out = lm_eval.simple_evaluate(model=lm, tasks=["wikitext"], limit=cfg["sanity_wikitext_docs"],
                                  log_samples=False, bootstrap_iters=0)
    r = out["results"]["wikitext"]
    return {"word_perplexity": r["word_perplexity,none"], "byte_perplexity": r["byte_perplexity,none"],
            "bits_per_byte": r["bits_per_byte,none"]}

In [ ]:
sanity_path = DRIVE / "sanity_wikitext.json"
if sanity_path.exists():
    sanity = json.loads(sanity_path.read_text())
else:
    sanity = {}
    tok = load_tokenizer()
    for base in ["fp16", "int4-rtn"]:
        model, qstats = load_base(base)
        t0 = time.time()
        sanity[base] = wikitext_ppl(model, tok)
        sanity[base]["seconds"] = round(time.time() - t0, 1)
        if qstats:
            sanity[base]["mean_weight_rel_err"] = float(np.mean([s["rel_err"] for s in qstats]))
            sanity[base]["n_quantized_tensors"] = len(qstats)
            sanity[base]["modes"] = sorted({(s["mode"], s["group_size"]) for s in qstats})
        del model; free_memory()
    sanity["config"] = {k: CONFIG[k] for k in ("sanity_wikitext_docs", "sanity_max_length", "eval_add_bos")}
    write_json(sanity_path, sanity)

for base in ["fp16", "int4-rtn"]:
    print(f"{base:9s} Colab fake-quant word ppl = {sanity[base]['word_perplexity']:.2f}   "
          f"(laptop OpenVINO: {PHASE1_WIKITEXT_PPL[base]})")
print("INT4 / FP16 ratio, Colab:", round(sanity["int4-rtn"]["word_perplexity"] / sanity["fp16"]["word_perplexity"], 3),
      "| laptop:", round(PHASE1_WIKITEXT_PPL["int4-rtn"] / PHASE1_WIKITEXT_PPL["fp16"], 3))

## 7. Training data: a fixed Dolly-15k subset

* Prompt template (Alpaca-style): instruction, then the optional context, then the response, followed by the EOS token.
* Examples longer than 512 tokens are **dropped** (not truncated), then the pool is shuffled with `data_seed=1234`. The first 200 examples form a small held-out set and the next 5,000 are the training set. **Every run sees the same 5,000 examples.** The training seed changes only the LoRA initialisation, dropout, and the order of batches.
* By default, loss is computed only on response tokens.
* The example IDs are saved to Drive (`data_subset.json`) so the subset is documented and reproducible.

In [ ]:
PROMPT_WITH_CONTEXT = "### Instruction:\n{instruction}\n\n### Context:\n{context}\n\n### Response:\n"
PROMPT_NO_CONTEXT = "### Instruction:\n{instruction}\n\n### Response:\n"

def build_prompt(ex):
    ctx = (ex.get("context") or "").strip()
    tpl = PROMPT_WITH_CONTEXT if ctx else PROMPT_NO_CONTEXT
    return tpl.format(instruction=ex["instruction"].strip(), context=ctx)

def tokenize_example(ex, tok, cfg=CONFIG):
    prompt_ids = tok(build_prompt(ex), add_special_tokens=True)["input_ids"]          # starts with BOS
    resp_ids = tok(ex["response"].strip(), add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    input_ids = prompt_ids + resp_ids
    labels = ([-100] * len(prompt_ids) + resp_ids) if cfg["loss_on_response_only"] else list(input_ids)
    return {"input_ids": input_ids, "labels": labels, "length": len(input_ids)}

def build_datasets(tok, cfg=CONFIG, raw=None):
    if raw is None:
        raw = datasets.load_dataset(cfg["dataset_id"], split="train")
    raw = raw.add_column("orig_idx", list(range(len(raw))))
    tokd = raw.map(lambda ex: tokenize_example(ex, tok, cfg), remove_columns=[c for c in raw.column_names if c != "orig_idx"])
    tokd = tokd.filter(lambda ex: ex["length"] <= cfg["max_seq_len"] and any(l != -100 for l in ex["labels"]))
    tokd = tokd.shuffle(seed=cfg["data_seed"])
    need = cfg["holdout_size"] + cfg["train_size"]
    if len(tokd) < need:
        raise ValueError(f"only {len(tokd)} usable examples, need {need}")
    holdout = tokd.select(range(cfg["holdout_size"]))
    train = tokd.select(range(cfg["holdout_size"], need))
    ids = {"train_orig_idx": [int(i) for i in train["orig_idx"]], "holdout_orig_idx": [int(i) for i in holdout["orig_idx"]]}
    ids["sha1"] = hashlib.sha1(json.dumps(ids["train_orig_idx"]).encode()).hexdigest()
    ids["train_tokens"] = int(sum(int(n) for n in train["length"]))
    return train.remove_columns(["orig_idx"]), holdout.remove_columns(["orig_idx"]), ids

class PadCollator:
    """Right-pads a batch to its longest example; padded labels are -100 so they never count in the loss."""
    def __init__(self, pad_id):
        self.pad_id = pad_id
    def __call__(self, feats):
        n = max(len(f["input_ids"]) for f in feats)
        ids = torch.full((len(feats), n), self.pad_id, dtype=torch.long)
        lab = torch.full((len(feats), n), -100, dtype=torch.long)
        att = torch.zeros((len(feats), n), dtype=torch.long)
        for i, f in enumerate(feats):
            L = len(f["input_ids"])
            ids[i, :L] = torch.tensor(f["input_ids"]); lab[i, :L] = torch.tensor(f["labels"]); att[i, :L] = 1
        return {"input_ids": ids, "labels": lab, "attention_mask": att}

In [ ]:
tok = load_tokenizer()
train_ds, holdout_ds, subset_ids = build_datasets(tok)
write_json(DRIVE / "data_subset.json", subset_ids)
print(f"train={len(train_ds)}  holdout={len(holdout_ds)}  train tokens={subset_ids['train_tokens']:,}  sha1={subset_ids['sha1'][:12]}")
print("--- example ---\n" + tok.decode(train_ds[0]["input_ids"])[:600])

## 8. One LoRA training run

`train_one(base, seed)` does the following:
1. Loads the fake-quant base (FP16, frozen).
2. Sets the seed, then attaches LoRA to q/k/v/o. The **adapter parameters are kept in FP32** because the T4 has no bf16, and fp16 mixed precision needs FP32 trainable weights for its gradient scaler. The forward pass runs under fp16 autocast.
3. Trains for 1 epoch with the Hugging Face `Trainer`.
4. Writes to `MyDrive/quant-study-lora/<base>-seed<k>/`: the adapter (`adapter_model.safetensors`, `adapter_config.json`), `run_config.json`, `train_log.csv` (loss/lr every 10 steps), `train_summary.json` (time, final loss, held-out loss, peak memory), and finally the `DONE` marker.

If Colab dies mid-run, that run has no `DONE` file and will simply be redone from scratch next time.

In [ ]:
class CSVLossLogger(TrainerCallback):
    """Appends each Trainer log line (step, loss, lr, grad_norm, elapsed) to a CSV."""
    def __init__(self, path):
        self.path, self.t0 = Path(path), time.time()
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text("step,epoch,loss,learning_rate,grad_norm,elapsed_s\n")
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            with self.path.open("a") as f:
                f.write(f"{state.global_step},{logs.get('epoch', '')},{logs['loss']},{logs.get('learning_rate', '')},"
                        f"{logs.get('grad_norm', '')},{time.time() - self.t0:.1f}\n")

def make_lora_model(model, seed, cfg=CONFIG):
    transformers.set_seed(seed)                      # controls LoRA init
    if cfg["gradient_checkpointing"]:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    lcfg = LoraConfig(r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"], lora_dropout=cfg["lora_dropout"],
                      target_modules=cfg["lora_target_modules"], bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, lcfg)
    for n, p in model.named_parameters():            # trainable = LoRA only, kept in fp32
        if p.requires_grad:
            p.data = p.data.float()
    return model

def train_one(base, seed, train_ds, holdout_ds, subset_ids, tok, cfg=CONFIG, out_dir=None):
    out = Path(out_dir) if out_dir else run_dir(base, seed)
    out.mkdir(parents=True, exist_ok=True)
    t_load = time.time()
    model, qstats = load_base(base, cfg)
    model = make_lora_model(model, seed, cfg)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    use_fp16 = cfg["fp16_autocast"] and torch.cuda.is_available()
    args = TrainingArguments(
        output_dir=str(Path("/tmp/trainer") / run_name(base, seed)),
        num_train_epochs=cfg["epochs"], learning_rate=cfg["learning_rate"], lr_scheduler_type=cfg["lr_scheduler"],
        warmup_ratio=cfg["warmup_ratio"], weight_decay=cfg["weight_decay"],
        per_device_train_batch_size=cfg["per_device_batch_size"], per_device_eval_batch_size=cfg["per_device_batch_size"],
        gradient_accumulation_steps=cfg["grad_accum_steps"], max_grad_norm=cfg["max_grad_norm"], optim=cfg["optimizer"],
        fp16=use_fp16, bf16=False, logging_steps=cfg["logging_steps"], save_strategy="no", report_to="none",
        seed=seed, data_seed=seed, dataloader_num_workers=0, remove_unused_columns=False,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=holdout_ds,
                      data_collator=PadCollator(tok.pad_token_id), callbacks=[CSVLossLogger(out / "train_log.csv")])
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    result = trainer.train()
    train_s = time.time() - t0
    holdout_loss = trainer.evaluate()["eval_loss"]
    model.save_pretrained(str(out))                  # adapter only; never merged into the quantized base
    summary = {
        "run": run_name(base, seed), "base": base, "seed": seed,
        "train_seconds": round(train_s, 1), "load_quant_seconds": round(t0 - t_load, 1),
        "global_steps": result.global_step, "train_loss_mean": result.training_loss,
        "final_logged_loss": next((h["loss"] for h in reversed(trainer.state.log_history) if "loss" in h), None),
        "holdout_loss": holdout_loss,
        "trainable_params": n_train, "total_params": n_all,
        "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None,
    }
    write_json(out / "train_summary.json", summary)
    write_json(out / "run_config.json", {"config": cfg, "base": base, "seed": seed, "env": env_info(),
                                         "data_subset_sha1": subset_ids["sha1"],
                                         "train_tokens": subset_ids["train_tokens"], "quant_stats": qstats})
    (out / "DONE").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
    del trainer, model; free_memory()
    return summary

## 9. Run the training grid (resumable)

Finished runs (folder contains `DONE`) are skipped. The order goes base by base, so an interrupted session still leaves complete seeds behind.
One run is roughly 10 to 20 minutes of training on a T4 plus about 2 minutes to load and quantize the model (see the README).

In [ ]:
bases = SESSION_BASES or CONFIG["bases"]
seeds = SESSION_SEEDS if SESSION_SEEDS is not None else CONFIG["seeds"]
for base in bases:
    for seed in seeds:
        name = run_name(base, seed)
        if is_done(run_dir(base, seed)):
            print(f"[skip] {name} already DONE"); continue
        print(f"[train] {name} ...")
        s = train_one(base, seed, train_ds, holdout_ds, subset_ids, tok)
        print(f"[done] {name}: {s['train_seconds']/60:.1f} min, train loss {s['train_loss_mean']:.4f}, "
              f"holdout loss {s['holdout_loss']:.4f}")

## 10. Evaluation with lm-evaluation-harness

`evaluate_one(base, adapter_dir)` builds the same fake-quant base, attaches the saved PEFT adapter (unmerged) if one is given, and runs lm-eval's `hf` model class on it in-process (zero-shot, full task sets).
It first evaluates the **three no-LoRA bases** (`<base>-nolora`), then all nine adapters. Each result goes to `MyDrive/quant-study-lora/eval/<name>.json`, and an existing file means "done, skip".
A full six-task evaluation takes roughly 25 to 40 minutes per model on a T4. HellaSwag (10k examples x 4 endings) is most of that time.

In [ ]:
EVAL_DIR = DRIVE / "eval"

def evaluate_one(base, adapter_dir=None, tok=None, cfg=CONFIG):
    tok = tok or load_tokenizer(cfg)
    model, _ = load_base(base, cfg)
    if adapter_dir is not None:
        model = PeftModel.from_pretrained(model, str(adapter_dir), is_trainable=False)
        model.eval()
    lm = make_lm(model, tok, cfg)
    t0 = time.time()
    out = lm_eval.simple_evaluate(model=lm, tasks=cfg["eval_tasks"], num_fewshot=cfg["eval_num_fewshot"],
                                  limit=cfg["eval_limit"], log_samples=False)
    secs = time.time() - t0
    metrics = {}
    for task, m in TASK_METRIC.items():
        if task in out["results"]:
            r = out["results"][task]
            metrics[task] = {"metric": m, "value": r[f"{m},none"], "stderr": r.get(f"{m}_stderr,none")}
    del lm, model; free_memory()
    return {"base": base, "adapter": str(adapter_dir) if adapter_dir else None, "metrics": metrics,
            "eval_seconds": round(secs, 1), "lm_eval_version": lm_eval.__version__,
            "eval_limit": cfg["eval_limit"], "num_fewshot": cfg["eval_num_fewshot"],
            "add_bos": cfg["eval_add_bos"], "env": env_info(),
            "raw_results": out["results"], "versions": out.get("versions"), "n-shot": out.get("n-shot")}

def eval_jobs(cfg=CONFIG):
    """No-LoRA bases first, then every finished adapter."""
    jobs = [(f"{b}-nolora", b, None) for b in cfg["bases"]]
    for b in cfg["bases"]:
        for s in cfg["seeds"]:
            jobs.append((run_name(b, s), b, run_dir(b, s)))
    return jobs

In [ ]:
tok = load_tokenizer()
for name, base, adir in eval_jobs():
    if SESSION_BASES and base not in SESSION_BASES:
        continue
    dest = EVAL_DIR / f"{name}.json"
    if dest.exists():
        print(f"[skip] {name} already evaluated"); continue
    if adir is not None and not is_done(adir):
        print(f"[wait] {name}: adapter not trained yet"); continue
    print(f"[eval] {name} ...")
    res = evaluate_one(base, adir, tok)
    write_json(dest, res)
    print(f"[done] {name} in {res['eval_seconds']/60:.1f} min:",
          {t: round(v["value"], 4) for t, v in res["metrics"].items()})

## 11. Aggregate results

For each base this shows the no-LoRA score (one number, since evaluation is deterministic) and the LoRA score as **mean ± sample std (ddof=1) over the 3 seeds**, per task and averaged over the six tasks. It also shows the change from LoRA (LoRA mean minus no-LoRA).
It also reports the share of the quantization gap that LoRA recovers: `(LoRA_q - noLoRA_q) / (noLoRA_fp16 - noLoRA_q)`. Treat this with care when the gap is tiny (e.g. INT8). For a cleaner comparison, also look at FP16+LoRA, because LoRA on Dolly can move scores even with no quantization.
The tables are saved to Drive as CSV and Markdown. Training time and loss per run are in `training_summary.csv`.

In [ ]:
import pandas as pd
from IPython.display import display

def load_eval(name):
    p = EVAL_DIR / f"{name}.json"
    return json.loads(p.read_text()) if p.exists() else None

tasks = CONFIG["eval_tasks"]
rows, long = [], []
for base in CONFIG["bases"]:
    nl = load_eval(f"{base}-nolora")
    if nl:
        vals = {t: nl["metrics"][t]["value"] for t in tasks}
        vals["avg"] = float(np.mean(list(vals.values())))
        rows.append({"base": base, "setting": "no-LoRA", "n": 1, **{t: f"{100*v:.2f}" for t, v in vals.items()}})
        long += [{"base": base, "setting": "no-LoRA", "seed": None, "task": t, "value": v} for t, v in vals.items()]
    per_seed = []
    for s in CONFIG["seeds"]:
        r = load_eval(run_name(base, s))
        if r:
            v = {t: r["metrics"][t]["value"] for t in tasks}
            v["avg"] = float(np.mean(list(v.values())))
            per_seed.append(v)
            long += [{"base": base, "setting": "LoRA", "seed": s, "task": t, "value": x} for t, x in v.items()]
    if per_seed:
        df = pd.DataFrame(per_seed)
        cell = lambda c: f"{100*df[c].mean():.2f} ± {100*df[c].std(ddof=1):.2f}" if len(df) > 1 else f"{100*df[c].mean():.2f}"
        rows.append({"base": base, "setting": "LoRA", "n": len(df), **{c: cell(c) for c in df.columns}})
        if nl:
            rows.append({"base": base, "setting": "Δ (LoRA − no-LoRA)", "n": len(df),
                         **{c: f"{100*(df[c].mean() - (vals[c])):+.2f}" for c in df.columns}})

table = pd.DataFrame(rows)
long_df = pd.DataFrame(long)
display(table)

# Gap recovery on the 6-task average.
if not long_df.empty:
    avg = long_df[long_df.task == "avg"].groupby(["base", "setting"])["value"].mean().unstack()
    if "no-LoRA" in avg and "fp16" in avg.index:
        for b in [x for x in CONFIG["bases"] if x != "fp16" and x in avg.index]:
            gap = avg.loc["fp16", "no-LoRA"] - avg.loc[b, "no-LoRA"]
            if "LoRA" in avg and not math.isnan(avg.loc[b, "LoRA"]) and abs(gap) > 1e-9:
                print(f"{b}: quantization gap {100*gap:.2f} pts; LoRA recovers {100*(avg.loc[b,'LoRA']-avg.loc[b,'no-LoRA'])/gap:.0f}% of it")

# Training summary (time + loss per run)
tr = [json.loads((run_dir(b, s) / "train_summary.json").read_text())
      for b in CONFIG["bases"] for s in CONFIG["seeds"] if (run_dir(b, s) / "train_summary.json").exists()]
train_df = pd.DataFrame(tr)
if not train_df.empty:
    display(train_df[["run", "train_seconds", "train_loss_mean", "final_logged_loss", "holdout_loss", "peak_gpu_mem_gb"]])
    train_df.to_csv(DRIVE / "training_summary.csv", index=False)

table.to_csv(DRIVE / "results_table.csv", index=False)
long_df.to_csv(DRIVE / "results_long.csv", index=False)
(DRIVE / "results_table.md").write_text(table.to_markdown(index=False) if hasattr(table, "to_markdown") else table.to_string())
print("Saved results_table.csv / results_long.csv / results_table.md / training_summary.csv to", DRIVE)

## 12. Re-score with per-question outputs (for the significance test)

The first evals saved only each model's final scores. The paired significance test needs to know which questions each model got right, so this re-runs the same evals for all 12 models (3 no-LoRA bases + 9 adapters) and saves per-question correctness to `quant-study-lora/eval_samples/`. No retraining.

* Run the notebook top-down: training and the original eval skip themselves because their results already exist on Drive.
* About 16-18 min per model on a T4, so about 3.5-4 h total. Resumable: if Colab disconnects, just run everything again and finished models are skipped.
* Each model prints `OK` per task if the re-score matches the saved accuracy. Tell me if you see `MISMATCH`.

In [ ]:
# ==== Re-score with per-example outputs (for the paired bootstrap) ====
# Paste into a new cell AFTER the cells that define CONFIG, DRIVE, load_base, make_lm,
# load_tokenizer, eval_jobs, is_done, TASK_METRIC (i.e. run the notebook top-down, skip training/eval).
# Re-runs lm-eval on all 12 models (3 no-LoRA + 9 adapters) with log_samples=True and saves ONLY
# compact per-question correctness to <drive_root>/eval_samples/<name>.json.gz (~100 KB each).
# Resumable: finished models are skipped. Also checks the re-scored accuracy matches eval/<name>.json.
# Time: ~16-18 min per model on a T4 (same as the original evals) -> ~3.5-4 h for all 12.
import gzip, json, time
import lm_eval
from peft import PeftModel

SAMPLE_DIR = DRIVE / "eval_samples"; SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
tok = load_tokenizer()
for name, base, adir in eval_jobs():
    dest = SAMPLE_DIR / f"{name}.json.gz"
    if dest.exists():
        print(f"[skip] {name}"); continue
    if adir is not None and not is_done(adir):
        print(f"[wait] {name}: adapter missing"); continue
    print(f"[rescore] {name} ...", flush=True); t0 = time.time()
    model, _ = load_base(base)
    if adir is not None:
        model = PeftModel.from_pretrained(model, str(adir), is_trainable=False); model.eval()
    lm = make_lm(model, tok)
    out = lm_eval.simple_evaluate(model=lm, tasks=CONFIG["eval_tasks"], num_fewshot=CONFIG["eval_num_fewshot"],
                                  limit=CONFIG["eval_limit"], log_samples=True, bootstrap_iters=0)
    per = {}
    for task, metric in TASK_METRIC.items():
        rows = sorted(out["samples"][task], key=lambda s: s["doc_id"])
        per[task] = {"metric": metric, "doc_id": [int(s["doc_id"]) for s in rows],
                     "correct": [int(round(float(s[metric]))) for s in rows]}
        acc = sum(per[task]["correct"]) / len(rows)
        ref = json.load(open(EVAL_DIR / f"{name}.json"))["metrics"][task]["value"]
        flag = "OK" if abs(acc - ref) < 1e-6 else f"MISMATCH (saved {ref:.4f})"
        print(f"   {task:13s} n={len(rows):5d} acc={acc:.4f} {flag}")
    with gzip.open(dest, "wt") as f:
        json.dump({"name": name, "base": base, "adapter": str(adir) if adir else None,
                   "lm_eval_version": lm_eval.__version__, "per_task": per}, f)
    del lm, model; free_memory()
    print(f"[done] {name} in {(time.time()-t0)/60:.1f} min", flush=True)
print("All done. Download the eval_samples folder (or it is already on Drive).")


## Download results (needed only if USE_DRIVE = False)
Zips everything saved so far and downloads it. Next session, upload the zip with the Files panel to `/content/` and run the unzip line before the training cells.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/quant-study-lora-results", "zip", CONFIG["drive_root"])
files.download(zip_path)
# To restore next session (before training):  !unzip -o -q /content/quant-study-lora-results.zip -d /content/quant-study-lora